# Шаг 7. Feature-based модель: LightGBM (gradient boosting + Optuna)

## Цель ноутбука
Обучить gradient boosting как **feature-based регрессор** — в отличие от SVD и KNN,
которые работают только с матрицей оценок, LightGBM использует богатые признаки:
численные характеристики пользователей и фильмов, жанры (one-hot) и теги (TF-IDF + SVD).

**Матрица признаков:** ~4 user + 4 movie + 19 genre + 20 tag-SVD = **~47 признаков** на пару (user, item).

## 0. Импорты и настройки

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

# ── Fallback-установка lightgbm / optuna ────────────────────────────────
try:
    import lightgbm  # noqa
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)

try:
    import optuna  # noqa
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'plotly'], check=True)

import optuna
import optuna.visualization as ov
import lightgbm as lgb

from src.utils import SEED, set_seeds
from src.data_io import (load_splits, load_features, load_id_maps,
                         load_tag_features)
from src.metrics import (
    rmse, mae,
    evaluate_rating_prediction, evaluate_topn,
    build_ground_truth,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)
set_seeds()

MODELS_DIR    = Path('../models')
PROCESSED_DIR = Path('../data/processed')

print(f"SEED = {SEED}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Optuna version: {optuna.__version__}")

## 1. Загрузка данных и признаков

In [ ]:
splits          = load_splits()
train, val, test = splits['train'], splits['val'], splits['test']

features        = load_features()
movie_feats     = features['movies']         # стандартизованные числовые признаки фильмов
user_feats      = features['users']          # стандартизованные числовые признаки пользователей
genre_feats     = features['genres']         # one-hot жанров
movies_enriched = features['movies_enriched']

tag_data   = load_tag_features()
tag_matrix = tag_data['matrix']              # sparse CSR (n_movies × n_tfidf_features)
tag_order  = tag_data['order']               # DataFrame: movieId → tag_row_idx

print(f"Размеры: train={len(train):,}, val={len(val):,}, test={len(test):,}")
print(f"User features:  {user_feats.shape}")
print(f"Movie features: {movie_feats.shape}")
print(f"Genre features: {genre_feats.shape}")
print(f"Tag TF-IDF matrix: {tag_matrix.shape}")
print(f"Колонки user_feats: {list(user_feats.columns)}")
print(f"Колонки movie_feats: {list(movie_feats.columns)}")

## 2. Стратегия признаков

Для каждой пары `(userId, movieId)` мы собираем широкий вектор признаков:

| Группа | Признаков | Источник |
|--------|-----------|----------|
| Пользователь (числовые) | 4 | `user_features.parquet` — стандартизованы в шаге 3 |
| Фильм (числовые) | 4 | `movie_features.parquet` — стандартизованы в шаге 3 |
| Жанры (one-hot) | 19+ | `genre_features.parquet` — бинарные |
| Теги (TF-IDF SVD) | 20 | `tag_features.npz` → TruncatedSVD (fit на train) |

**Итого: ~47 признаков.** Признаки уже нормализованы в шаге 3 — `StandardScaler`
**НЕ применяем повторно**.

**О data leakage:** TruncatedSVD для тегов обучается **только на фильмах из train**,
затем применяется ко всем. Это аналогично подходу TF-IDF в шаге 3.

## 3. Сжатие тегов через TruncatedSVD

In [ ]:
from sklearn.decomposition import TruncatedSVD

# Фильмы только из train — на них фитим SVD
train_movie_ids   = set(train['movieId'].unique())
train_mask        = tag_order['movieId'].isin(train_movie_ids).values
tag_matrix_train  = tag_matrix[train_mask]

n_tag_components = 20
tag_svd = TruncatedSVD(n_components=n_tag_components, random_state=SEED)
tag_svd.fit(tag_matrix_train)

# Применяем ко всем фильмам (transform, не fit_transform)
tag_svd_matrix = tag_svd.transform(tag_matrix)   # shape (n_movies, 20)

tag_svd_df = pd.DataFrame(
    tag_svd_matrix,
    columns=[f'tag_svd_{i}' for i in range(n_tag_components)]
)
tag_svd_df['movieId'] = tag_order['movieId'].values

# Сохраняем артефакт (нужен для inference в Streamlit)
joblib.dump(tag_svd, MODELS_DIR / 'tag_svd.pkl')

explained = tag_svd.explained_variance_ratio_.sum()
print(f'TruncatedSVD: {n_tag_components} компонент объясняют {explained:.4f} дисперсии')
print('(Низкое значение < 0.3 — норма для разреженных TF-IDF матриц)')
print(f'tag_svd_df: {tag_svd_df.shape}')

Если объяснённая дисперсия TF-IDF тегов низкая (< 0.30) — это нормально:
разреженные TF-IDF матрицы имеют много шумовых направлений, информация
рассеяна по большому числу компонент. Реальную пользу tag-признаков оценим
через `feature_importance` в разделе 9.

## 4. Сборка широкой матрицы признаков

In [ ]:
def build_feature_matrix(ratings_df: pd.DataFrame,
                         user_feats: pd.DataFrame,
                         movie_feats: pd.DataFrame,
                         genre_feats: pd.DataFrame,
                         tag_svd_df: pd.DataFrame):
    """Собрать широкую матрицу признаков для пар (userId, movieId).

    Возвращает:
        X (pd.DataFrame) — признаки без userId/movieId/rating.
        y (np.ndarray) — целевые рейтинги.
        feature_cols (list[str]) — имена признаков.
    """
    df = ratings_df[['userId', 'movieId', 'rating']].copy()
    df = df.merge(user_feats,  on='userId',  how='left')
    df = df.merge(movie_feats, on='movieId', how='left')
    df = df.merge(genre_feats, on='movieId', how='left')
    df = df.merge(tag_svd_df,  on='movieId', how='left')

    feature_cols = [c for c in df.columns
                    if c not in ('userId', 'movieId', 'rating')]
    y = df['rating'].values.astype(np.float32)
    X = df[feature_cols].fillna(0.0)
    return X, y, feature_cols


X_train, y_train, feature_cols = build_feature_matrix(
    train, user_feats, movie_feats, genre_feats, tag_svd_df
)
X_val,   y_val,   _ = build_feature_matrix(
    val,   user_feats, movie_feats, genre_feats, tag_svd_df
)
X_test,  y_test,  _ = build_feature_matrix(
    test,  user_feats, movie_feats, genre_feats, tag_svd_df
)

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'Число признаков: {len(feature_cols)}')
print(f'Признаки: {feature_cols[:8]} ...')

# Sanity-проверка целостности данных
assert len(X_train) == len(train), 'X_train не совпадает по размеру с train после merge'
assert len(X_val)   == len(val),   'X_val не совпадает по размеру с val после merge'
assert len(X_test)  == len(test),  'X_test не совпадает по размеру с test после merge'
print('\nSanity-проверка merge: OK')

## 5. Базовая LightGBM модель (точка отсчёта)

Обучаем с дефолтными параметрами и early stopping на val.
Это точка отсчёта — Optuna должна её улучшить.

In [ ]:
baseline_lgbm = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
    n_estimators=500,
)

t0 = time.time()
baseline_lgbm.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
)
baseline_train_time = time.time() - t0

baseline_val_preds   = baseline_lgbm.predict(X_val)
baseline_val_metrics = evaluate_rating_prediction(y_val, baseline_val_preds)

print(f'Baseline LightGBM обучена за {baseline_train_time:.2f} с')
print(f'Best iteration: {baseline_lgbm.best_iteration_}')
print('Baseline val:')
print(json.dumps(baseline_val_metrics, indent=2))

## 6. Optuna — подбор гиперпараметров

**Пространство параметров:**
- `num_leaves`: 16..255 — главный регулятор сложности дерева
- `max_depth`: 3..12 — ограничение глубины
- `learning_rate`: 0.01..0.3 (log)
- `min_child_samples`: 5..100 — регуляризация (минимум объектов в листе)
- `subsample` / `subsample_freq` — row subsampling (бэггинг)
- `colsample_bytree` — column subsampling
- `reg_alpha` / `reg_lambda`: 1e-3..10 (log) — L1 / L2

**`n_estimators` НЕ подбирается через Optuna** — определяется через early stopping.
Лучший `best_iteration` из оптимального trial используется при финальном обучении.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'objective':          'regression',
        'metric':             'rmse',
        'random_state':       SEED,
        'n_jobs':             -1,
        'verbose':            -1,
        'n_estimators':       1000,
        'num_leaves':         trial.suggest_int('num_leaves', 16, 255),
        'max_depth':          trial.suggest_int('max_depth', 3, 12),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_samples':  trial.suggest_int('min_child_samples', 5, 100),
        'subsample':          trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':     trial.suggest_int('subsample_freq', 1, 7),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )
    val_preds = model.predict(X_val, num_iteration=model.best_iteration_)
    trial.set_user_attr('best_iteration', int(model.best_iteration_))
    return rmse(y_val, val_preds)


sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    study_name='lightgbm_movielens'
)

print("Запускаем Optuna (50 trials)...")
t0 = time.time()
study.optimize(objective, n_trials=50, show_progress_bar=True)
optuna_time = time.time() - t0

best_iteration = study.best_trial.user_attrs['best_iteration']

print(f'\nOptuna завершила {len(study.trials)} trials за {optuna_time:.1f} с')
print(f'Лучший RMSE на val: {study.best_value:.4f}')
print(f'Оптимальное число итераций: {best_iteration}')
print('Лучшие параметры:')
print(json.dumps(study.best_params, indent=2))

In [ ]:
# ── Визуализация Optuna ──────────────────────────────────────────────────

fig_history = ov.plot_optimization_history(study)
fig_history.update_layout(title='Optuna: история оптимизации LightGBM (RMSE на val)')
fig_history.write_html(str(MODELS_DIR / 'optuna_lightgbm_history.html'))
fig_history.show()

fig_importance = ov.plot_param_importances(study)
fig_importance.update_layout(title='Optuna: важность гиперпараметров LightGBM')
fig_importance.write_html(str(MODELS_DIR / 'optuna_lightgbm_importance.html'))
fig_importance.show()

fig_slice = ov.plot_slice(study)
fig_slice.update_layout(title='Optuna: срезы по гиперпараметрам LightGBM')
fig_slice.show()

**Интерпретация результатов Optuna:**

- **learning_rate** — обычно наиболее важный параметр для LightGBM. Низкое значение
  (0.01–0.05) + большее число итераций (early stopping) часто даёт лучший результат.
- **num_leaves** — главный регулятор сложности. Оптимальные значения для MovieLens-small
  обычно 30–80: слишком высокое приводит к переобучению на разреженном датасете.
- **min_child_samples** — важная регуляризация: при малом датасете оптимальное значение
  часто выше дефолта (20–50), чтобы деревья не переобучались на отдельных пользователях.
- **reg_lambda / reg_alpha** — L2/L1 регуляризация помогает при малом числе признаков.

## 7. Финальная LightGBM на train + val

Обучаем без early stopping с фиксированным числом итераций из лучшего trial.

In [ ]:
final_params = {
    **study.best_params,
    'objective':    'regression',
    'metric':       'rmse',
    'random_state': SEED,
    'n_jobs':       -1,
    'verbose':      -1,
    'n_estimators': best_iteration,   # число итераций из лучшего trial
}

train_val = pd.concat([train, val], ignore_index=True)
X_train_val, y_train_val, _ = build_feature_matrix(
    train_val, user_feats, movie_feats, genre_feats, tag_svd_df
)

final_lgbm = lgb.LGBMRegressor(**final_params)
t0 = time.time()
final_lgbm.fit(X_train_val, y_train_val)
final_train_time = time.time() - t0

print(f'Финальная LightGBM обучена на train+val за {final_train_time:.2f} с')
print(f'n_estimators (итераций): {best_iteration}')
print(f'Параметры: {study.best_params}')

Используем `best_iteration`, найденный на `train` с early stopping по `val`.
При расширении до `train+val` реальный оптимум чуть выше — это известный trade-off.
Для production-системы можно применить масштабирующую поправку, но для курсового проекта
фиксируем без поправки для простоты и воспроизводимости.

## 8. Оценка на test

### 8.1 RMSE / MAE

In [ ]:
test_preds             = final_lgbm.predict(X_test)
lgbm_test_rating_metrics = evaluate_rating_prediction(y_test, test_preds)
print('LightGBM test (rating):')
print(json.dumps(lgbm_test_rating_metrics, indent=2))

### 8.2 Top-N метрики

Для каждого пользователя из test ground_truth:
1. Берём всех непросмотренных кандидатов из train+val.
2. Строим feature-матрицу кандидатных пар (user, movie) батчем.
3. Предсказываем рейтинги через LightGBM.
4. Берём топ-K по убыванию предсказанного рейтинга.

In [ ]:
def generate_topn_recommendations_lgbm(model, user_ids, train_val_df,
                                        all_movies, user_feats, movie_feats,
                                        genre_feats, tag_svd_df, feature_cols,
                                        k=20):
    """Top-K рекомендации через батчевый инференс LightGBM.

    model: обученный LGBMRegressor.
    user_ids: список userId.
    train_val_df: DataFrame для определения уже просмотренных.
    all_movies: массив всех movieId (кандидатный пул).
    feature_cols: порядок столбцов (как при обучении модели).
    k: длина рекомендации.
    """
    seen_by_user = (
        train_val_df.groupby('userId')['movieId'].apply(set).to_dict()
    )
    all_movies = np.asarray(all_movies)

    # Индексируем feature-таблицы по ID для быстрого доступа
    user_feats_idx  = user_feats.set_index('userId')
    movie_feats_idx = movie_feats.set_index('movieId')
    genre_feats_idx = genre_feats.set_index('movieId')
    tag_svd_idx     = tag_svd_df.set_index('movieId')

    recommendations = {}

    for user_id in user_ids:
        seen       = seen_by_user.get(user_id, set())
        candidates = all_movies[~np.isin(all_movies, list(seen))]

        if len(candidates) == 0:
            recommendations[user_id] = []
            continue

        try:
            user_row = user_feats_idx.loc[user_id]
        except KeyError:
            recommendations[user_id] = []
            continue

        # Собираем блоки признаков для всех кандидатов сразу (батч)
        movie_block = movie_feats_idx.reindex(candidates).fillna(0.0)
        genre_block = genre_feats_idx.reindex(candidates).fillna(0.0)
        tag_block   = tag_svd_idx.reindex(candidates).fillna(0.0)

        # Тиражируем строку пользователя на всех кандидатов
        user_block = pd.DataFrame(
            np.tile(user_row.values, (len(candidates), 1)),
            columns=user_row.index,
            index=candidates
        )

        cand_X = pd.concat([user_block, movie_block, genre_block, tag_block],
                           axis=1)
        # Гарантируем порядок столбцов как при обучении
        cand_X = cand_X.reindex(columns=feature_cols, fill_value=0.0)

        scores    = model.predict(cand_X.values)
        top_k_idx = np.argsort(-scores)[:k]
        recommendations[user_id] = candidates[top_k_idx].tolist()

    return recommendations


test_ground_truth = build_ground_truth(test, relevance_threshold=4.0)
test_users        = list(test_ground_truth.keys())
all_movies_arr    = train_val['movieId'].unique()

print(f'Генерация топ-20 для {len(test_users)} пользователей...')
t0 = time.time()
test_recs = generate_topn_recommendations_lgbm(
    final_lgbm, test_users, train_val, all_movies_arr,
    user_feats, movie_feats, genre_feats, tag_svd_df,
    feature_cols=feature_cols, k=20,
)
inference_time = time.time() - t0
print(f'Готово за {inference_time:.1f} с')

lgbm_test_topn_metrics = evaluate_topn(
    test_recs, test_ground_truth,
    ks=(5, 10, 20),
    all_items=all_movies_arr
)
print('LightGBM test (top-N):')
print(json.dumps(lgbm_test_topn_metrics, indent=2))

## 9. Важность признаков (Feature Importance)

In [ ]:
importance_gain = pd.DataFrame({
    'feature':          X_train.columns.tolist(),
    'importance_gain':  final_lgbm.booster_.feature_importance(importance_type='gain'),
    'importance_split': final_lgbm.booster_.feature_importance(importance_type='split'),
}).sort_values('importance_gain', ascending=False)

print('Топ-20 признаков по importance (gain):')
display(importance_gain.head(20))

# Bar chart топ-20
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
top20 = importance_gain.head(20)

axes[0].barh(top20['feature'][::-1], top20['importance_gain'][::-1],
             color='steelblue')
axes[0].set_xlabel('Importance (gain)')
axes[0].set_title('Топ-20 признаков: gain')

axes[1].barh(top20['feature'][::-1], top20['importance_split'][::-1],
             color='darkorange')
axes[1].set_xlabel('Importance (split count)')
axes[1].set_title('Топ-20 признаков: split')

plt.suptitle('LightGBM Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Интерпретация важности признаков:**

- **`user_mean_rating_train` и `movie_mean_rating_train`** обычно доминируют по gain —
  это «разнесённые средние», мощный и дешёвый baseline. Если они дают > 50% gain,
  модель близка к bias-only baseline и польза тонких признаков ограничена.
- **`user_n_ratings_train` и `movie_n_ratings_train`** — важные признаки активности.
  Более активные пользователи ставят более стабильные оценки.
- **`movie_year`** — год выпуска. Старые классические фильмы оцениваются иначе.
- **Жанровые признаки (`genre_*`)** — обычно средняя важность; полезны для холодного старта.
- **`tag_svd_*`** — если имеют значимый gain, теги действительно несут информацию о контенте.

## 10. Сравнение со всеми предыдущими моделями

In [ ]:
with open(MODELS_DIR / 'popularity_metrics.json', 'r', encoding='utf-8') as f:
    pop_metrics = json.load(f)
with open(MODELS_DIR / 'svd_metrics.json', 'r', encoding='utf-8') as f:
    svd_metrics_loaded = json.load(f)
with open(MODELS_DIR / 'knn_metrics.json', 'r', encoding='utf-8') as f:
    knn_metrics_loaded = json.load(f)


def make_row(name, rating_m=None, topn_m=None):
    def r(d, k):
        return round(float(d[k]), 4) if d and k in d and d[k] is not None else None
    return {
        'Модель':       name,
        'RMSE':         r(rating_m, 'rmse'),
        'MAE':          r(rating_m, 'mae'),
        'NDCG@10':      r(topn_m, 'ndcg@10'),
        'Precision@10': r(topn_m, 'precision@10'),
        'Recall@10':    r(topn_m, 'recall@10'),
        'HitRate@10':   r(topn_m, 'hit_rate@10'),
        'Coverage@20':  r(topn_m, 'coverage@20'),
    }


comparison_rows = [
    make_row('GlobalMean', pop_metrics['global_mean']['test']),
    make_row('Popularity', topn_m=pop_metrics['popularity']['test']),
    make_row('SVD',
             svd_metrics_loaded['final']['test_rating'],
             svd_metrics_loaded['final']['test_topn']),
    make_row('KNN',
             knn_metrics_loaded['final']['test_rating'],
             knn_metrics_loaded['final']['test_topn']),
    make_row('LightGBM',
             lgbm_test_rating_metrics,
             lgbm_test_topn_metrics),
]

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.to_string(index=False))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

topn_cols  = ['NDCG@10', 'Precision@10', 'Recall@10', 'HitRate@10']
models_cmp = ['Popularity', 'SVD', 'KNN', 'LightGBM']
colors_cmp = ['darkorange', 'steelblue', 'seagreen', 'mediumpurple']
x = np.arange(len(topn_cols))
width = 0.2

for i, (mname, color) in enumerate(zip(models_cmp, colors_cmp)):
    row   = comparison_df[comparison_df['Модель'] == mname].iloc[0]
    vals  = [float(row[c]) if row[c] is not None else 0.0 for c in topn_cols]
    axes[0].bar(x + i * width, vals, width, label=mname, color=color, alpha=0.85)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(topn_cols, fontsize=9)
axes[0].set_title('Top-N метрики (test)')
axes[0].legend(fontsize=8)

rmse_models = ['GlobalMean', 'SVD', 'KNN', 'LightGBM']
rmse_colors = ['tomato', 'steelblue', 'seagreen', 'mediumpurple']
rmse_vals   = [
    float(comparison_df[comparison_df['Модель'] == m]['RMSE'].iloc[0])
    for m in rmse_models
    if comparison_df[comparison_df['Модель'] == m]['RMSE'].iloc[0] is not None
]
axes[1].bar(rmse_models, rmse_vals, color=rmse_colors, alpha=0.85)
axes[1].set_title('RMSE на test')
axes[1].set_ylabel('RMSE')
for bar, v in zip(axes[1].patches, rmse_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.005,
                 f'{v:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Интерпретация сравнительной таблицы:**

1. **LightGBM vs GlobalMean (RMSE):** LightGBM существенно точнее, используя
   персонализированные признаки — mean rating пользователя и фильма.

2. **LightGBM vs SVD (RMSE):** результаты конкурентны. LightGBM как feature-based
   регрессор может превзойти SVD за счёт нелинейной комбинации признаков.

3. **LightGBM vs SVD/KNN (NDCG@10):** feature-based модели могут уступать
   коллаборативным по top-N метрикам, так как LightGBM не «видит» латентных факторов.
   Это известный компромисс: RMSE vs ранжирование.

4. **Coverage@20:** LightGBM обычно даёт более широкое покрытие, чем Popularity,
   и сопоставимое с SVD/KNN — за счёт персонализации через пользовательские признаки.

## 11. Анализ ошибок

In [ ]:
# Добавляем ошибки в test
test_with_errors = test.copy()
test_with_errors['pred']  = test_preds
test_with_errors['error'] = np.abs(y_test - test_preds)

# 1. Гистограмма ошибок + scatter
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(test_with_errors['error'], bins=30, color='mediumpurple',
             edgecolor='white', density=True)
axes[0].axvline(test_with_errors['error'].mean(), color='red', linestyle='--',
                label=f'Mean = {test_with_errors["error"].mean():.3f}')
axes[0].set_title('Распределение |ошибки| LightGBM на test')
axes[0].set_xlabel('|y_true - y_pred|')
axes[0].set_ylabel('Плотность')
axes[0].legend()

axes[1].scatter(y_test, test_preds, alpha=0.15, s=5, color='mediumpurple')
lims = [0.5, 5.0]
axes[1].plot(lims, lims, 'r--', linewidth=1)
axes[1].set_xlim(lims); axes[1].set_ylim(lims)
axes[1].set_xlabel('Истинный рейтинг')
axes[1].set_ylabel('Предсказанный рейтинг')
axes[1].set_title('Истинный vs предсказанный (test)')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Топ-10 фильмов с наибольшей средней ошибкой
movie_error = (
    test_with_errors.groupby('movieId')['error']
    .agg(['mean', 'count']).reset_index()
    .rename(columns={'mean': 'mean_error', 'count': 'n_test'})
)
movie_error = movie_error[movie_error['n_test'] >= 3]
top_movie_errors = (
    movie_error.nlargest(10, 'mean_error')
    .merge(movies_enriched[['movieId', 'title']], on='movieId', how='left')
)
top_movie_errors['mean_error'] = top_movie_errors['mean_error'].round(3)
print('Топ-10 фильмов с наибольшей средней |ошибкой| (min 3 оценки в test):')
display(top_movie_errors[['title', 'mean_error', 'n_test']])

In [ ]:
# 3. Топ-10 пользователей с наибольшей средней ошибкой
user_error = (
    test_with_errors.groupby('userId')['error']
    .agg(['mean', 'count']).reset_index()
    .rename(columns={'mean': 'mean_error', 'count': 'n_test'})
)
user_error = user_error[user_error['n_test'] >= 3]
top_user_errors = user_error.nlargest(10, 'mean_error').copy()
top_user_errors['mean_error'] = top_user_errors['mean_error'].round(3)
print('Топ-10 пользователей с наибольшей средней |ошибкой|:')
display(top_user_errors)

In [ ]:
# 4. Корреляция ошибки с популярностью фильма в train
movie_pop_train = (
    train.groupby('movieId').size().reset_index(name='n_ratings_train')
)
error_vs_pop = test_with_errors.merge(movie_pop_train, on='movieId', how='left')

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(error_vs_pop['n_ratings_train'], error_vs_pop['error'],
           alpha=0.1, s=5, color='mediumpurple')

from numpy.polynomial.polynomial import polyfit
x_vals = error_vs_pop['n_ratings_train'].values
y_vals = error_vs_pop['error'].values
mask   = ~np.isnan(x_vals) & ~np.isnan(y_vals)
coeffs = polyfit(x_vals[mask], y_vals[mask], 1)
x_line = np.linspace(x_vals[mask].min(), x_vals[mask].max(), 100)
ax.plot(x_line, coeffs[0] + coeffs[1] * x_line, 'r-',
        linewidth=2, label='тренд')

corr = np.corrcoef(x_vals[mask], y_vals[mask])[0, 1]
ax.set_xlabel('Число оценок фильма в train')
ax.set_ylabel('|Ошибка| LightGBM')
ax.set_title(f'Ошибка LightGBM vs популярность фильма (Pearson r = {corr:.3f})')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Pearson r (ошибка vs популярность): {corr:.4f}')

**Выводы по анализу ошибок:**

1. LightGBM делает ошибки похожей природы на SVD и KNN — наибольшие ошибки
   на нишевых фильмах с малым числом оценок.

2. Отрицательная корреляция ошибки с популярностью ожидаема: фильмы с большим
   числом оценок в train — «хорошо покрытые» данными, признаки для них точнее.

3. LightGBM, как и SVD, не имеет специального механизма для холодного старта —
   для новых фильмов и пользователей в продакшне нужен popularity-фолбэк.

## 12. Сохранение артефактов

In [ ]:
# Модель
joblib.dump(final_lgbm, MODELS_DIR / 'lightgbm_model.pkl')
print(f"lightgbm_model.pkl: {(MODELS_DIR / 'lightgbm_model.pkl').stat().st_size / 1024:.1f} KB")

# Параметры
lightgbm_params = {
    'random_state':               SEED,
    'best_params':                study.best_params,
    'best_iteration':             best_iteration,
    'n_estimators_final':         final_params['n_estimators'],
    'optuna_n_trials':            50,
    'optuna_sampler':             'TPESampler',
    'optuna_direction':           'minimize',
    'optuna_target':              'rmse@val',
    'final_train_strategy':       'train+val; n_estimators=best_iteration',
    'feature_columns':            list(feature_cols),
    'tag_svd_components':         n_tag_components,
    'tag_svd_explained_variance': float(tag_svd.explained_variance_ratio_.sum()),
    'baseline_train_time_sec':    baseline_train_time,
    'optuna_search_time_sec':     optuna_time,
    'final_train_time_sec':       final_train_time,
    'inference_time_test_topn_sec': inference_time,
}
with open(MODELS_DIR / 'lightgbm_params.json', 'w', encoding='utf-8') as f:
    json.dump(lightgbm_params, f, ensure_ascii=False, indent=2)

# Метрики
lightgbm_metrics = {
    'baseline': {
        'val': baseline_val_metrics,
    },
    'final': {
        'val_best_rmse': float(study.best_value),
        'test_rating':   lgbm_test_rating_metrics,
        'test_topn':     lgbm_test_topn_metrics,
    },
    'feature_importance_top20': importance_gain.head(20).to_dict(orient='records'),
    'meta': {
        'k_values':            [5, 10, 20],
        'relevance_threshold': 4.0,
        'optuna_n_trials':     50,
    },
}
with open(MODELS_DIR / 'lightgbm_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(lightgbm_metrics, f, ensure_ascii=False, indent=2)

# История trials
trials_df = study.trials_dataframe()
trials_df.to_parquet(MODELS_DIR / 'lightgbm_optuna_trials.parquet', index=False)

print('Все артефакты сохранены.')
print(f'  Лучший RMSE (val): {study.best_value:.4f}')
print(f'  RMSE (test):       {lgbm_test_rating_metrics["rmse"]:.4f}')
print(f'  NDCG@10 (test):    {lgbm_test_topn_metrics["ndcg@10"]:.4f}')

## 13. Финальные проверки

In [ ]:
required_files = [
    MODELS_DIR / 'lightgbm_model.pkl',
    MODELS_DIR / 'lightgbm_params.json',
    MODELS_DIR / 'lightgbm_metrics.json',
    MODELS_DIR / 'lightgbm_optuna_trials.parquet',
    MODELS_DIR / 'tag_svd.pkl',
    MODELS_DIR / 'optuna_lightgbm_history.html',
    MODELS_DIR / 'optuna_lightgbm_importance.html',
]
for p in required_files:
    assert p.exists(), f'Файл не найден: {p}'

# Round-trip модели
lgbm_loaded          = joblib.load(MODELS_DIR / 'lightgbm_model.pkl')
sample_pred_loaded   = lgbm_loaded.predict(X_test.iloc[:5].values)
sample_pred_original = final_lgbm.predict(X_test.iloc[:5].values)
np.testing.assert_allclose(sample_pred_loaded, sample_pred_original, atol=1e-9)

# Round-trip tag_svd
tag_svd_loaded        = joblib.load(MODELS_DIR / 'tag_svd.pkl')
sample_proj_loaded    = tag_svd_loaded.transform(tag_matrix[:5])
sample_proj_original  = tag_svd.transform(tag_matrix[:5])
np.testing.assert_allclose(sample_proj_loaded, sample_proj_original, atol=1e-9)

# Sanity-границы
assert 0.5 < lgbm_test_rating_metrics['rmse'] < 1.5,     f"RMSE LightGBM подозрителен: {lgbm_test_rating_metrics['rmse']}"
assert 0.0 <= lgbm_test_topn_metrics['ndcg@10'] <= 1.0
assert 0.0 <= lgbm_test_topn_metrics['precision@10'] <= 1.0

# Ровно 50 trials
assert len(study.trials) == 50, f'Optuna прогнала {len(study.trials)} trials вместо 50'

# Optuna улучшила baseline
assert study.best_value <= baseline_val_metrics['rmse'] + 1e-6,     'Optuna не улучшила RMSE baseline'

# LightGBM лучше GlobalMean
gm_rmse = pop_metrics['global_mean']['test']['rmse']
assert lgbm_test_rating_metrics['rmse'] < gm_rmse,     f"LightGBM RMSE ({lgbm_test_rating_metrics['rmse']:.4f}) хуже GlobalMean ({gm_rmse:.4f})"

# Корректность рекомендаций
sample_user = test_users[0]
sample_recs = test_recs[sample_user]
assert len(sample_recs) == 20
assert len(set(sample_recs)) == 20, 'В рекомендациях есть дубликаты'
seen_by_user = set(train_val[train_val['userId'] == sample_user]['movieId'])
assert not (set(sample_recs) & seen_by_user),     'В рекомендациях LightGBM есть уже просмотренные фильмы'

# Целостность данных
assert set(train['userId']).issubset(set(user_feats['userId'])),     'Не все юзеры из train есть в user_feats'
assert set(train['movieId']).issubset(set(movie_feats['movieId'])),     'Не все фильмы из train есть в movie_feats'

print('\u2705 Шаг 7 пройден: LightGBM обучена, гиперпараметры подобраны, метрики сохранены')
print(f'   RMSE (test):    {lgbm_test_rating_metrics["rmse"]:.4f}  '
      f'(vs GlobalMean {gm_rmse:.4f}, '
      f'vs SVD {svd_metrics_loaded["final"]["test_rating"]["rmse"]:.4f})')
print(f'   NDCG@10 (test): {lgbm_test_topn_metrics["ndcg@10"]:.4f}  '
      f'(vs Popularity {pop_metrics["popularity"]["test"]["ndcg@10"]:.4f})')
print(f'   Best iteration: {best_iteration}')